In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# ---- load one season with both results and Pinnacle closing odds ----
df = pd.read_csv("data/E2_24-25.csv")
df = df[["Date","HomeTeam","AwayTeam","FTHG","FTAG","PSCH","PSCD","PSCA"]].copy()
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.dropna(subset=["PSCH","PSCD","PSCA"]).sort_values("Date").reset_index(drop=True)

# ---- train/test split (same 70/30 as always) ----
split = int(len(df) * 0.70)
train, test = df.iloc[:split].copy(), df.iloc[split:].copy()

# ---- MODEL side: shrunk strengths from train only (k=5) ----
k = 5.0
league_home, league_away = train["FTHG"].mean(), train["FTAG"].mean()
hs  = train.groupby("HomeTeam").agg(goals=("FTHG","sum"), conc=("FTAG","sum"), n=("FTHG","size"))
as_ = train.groupby("AwayTeam").agg(goals=("FTAG","sum"), conc=("FTHG","sum"), n=("FTAG","size"))
def shrink(total, n, lg): return (total + k*lg) / (n + k)
h_scored   = {t: shrink(r.goals, r.n, league_home) for t,r in hs.iterrows()}
h_conceded = {t: shrink(r.conc,  r.n, league_away) for t,r in hs.iterrows()}
a_scored   = {t: shrink(r.goals, r.n, league_away) for t,r in as_.iterrows()}
a_conceded = {t: shrink(r.conc,  r.n, league_home) for t,r in as_.iterrows()}

def model_probs(ht, at):
    he = h_scored[ht] * a_conceded[at] / league_home
    ae = a_scored[at] * h_conceded[ht] / league_away
    hw,dr,aw = 0.0,0.0,0.0
    for hg in range(10):
        for ag in range(10):
            p = poisson.pmf(hg,he)*poisson.pmf(ag,ae)
            if hg>ag: hw+=p
            elif hg==ag: dr+=p
            else: aw+=p
    return hw,dr,aw

# ---- MARKET side: margin-stripped (proportional) from Pinnacle closing ----
def market_probs(row):
    M = (1/row.PSCH + 1/row.PSCD + 1/row.PSCA) - 1
    fair = lambda O: (3*O)/(3 - M*O)
    return 1/fair(row.PSCH), 1/fair(row.PSCD), 1/fair(row.PSCA)

# ---- align both on each TEST match ----
rows = []
for _, r in test.iterrows():
    ht, at = r["HomeTeam"], r["AwayTeam"]
    if ht not in h_scored or at not in a_scored:
        continue
    mh, md, ma = model_probs(ht, at)
    kh, kd, ka = market_probs(r)
    actual = "H" if r.FTHG>r.FTAG else ("A" if r.FTHG<r.FTAG else "D")
    rows.append({
        "home":ht, "away":at, "actual":actual,
        "m_home":mh, "m_draw":md, "m_away":ma,      # model
        "k_home":kh, "k_draw":kd, "k_away":ka,      # market (true)
        "o_home":r.PSCH, "o_draw":r.PSCD, "o_away":r.PSCA  # raw odds for betting
    })

compare = pd.DataFrame(rows)
print(f"Test matches with both model and market: {len(compare)}")
compare[["home","away","actual","m_home","k_home","m_away","k_away"]].head()

Test matches with both model and market: 166


,home,away,actual,m_home,k_home,m_away,k_away
0,Northampton,Barnsley,A,0.253670,0.233557,0.494957,0.504167
1,Wigan,Reading,A,0.405049,0.493927,0.318785,0.233227
2,Stockport,Blackpool,H,0.466192,0.484041,0.294491,0.246945
3,Stevenage,Huddersfield,A,0.268957,0.360304,0.460317,0.357540
4,Peterboro,Shrewsbury,H,0.522788,0.494063,0.255473,0.245516


In [3]:
# edge = model probability / market "true" probability
# > 1.0 means model sees the outcome as underpriced (value)
compare["edge_home"] = compare["m_home"] / compare["k_home"]
compare["edge_draw"] = compare["m_draw"] / compare["k_draw"]
compare["edge_away"] = compare["m_away"] / compare["k_away"]

# how often does the model see "value" (edge > 1) on each outcome?
print("Outcomes where model sees value (edge > 1.0):")
print(f"  Home: {(compare['edge_home'] > 1).sum()} / {len(compare)}")
print(f"  Draw: {(compare['edge_draw'] > 1).sum()} / {len(compare)}")
print(f"  Away: {(compare['edge_away'] > 1).sum()} / {len(compare)}")

# distribution of edges — how big are the disagreements?
print("\nEdge summary (1.0 = agree with market):")
print(compare[["edge_home","edge_draw","edge_away"]].describe().round(3))

Outcomes where model sees value (edge > 1.0):
  Home: 81 / 166
  Draw: 66 / 166
  Away: 94 / 166

Edge summary (1.0 = agree with market):
       edge_home  edge_draw  edge_away
count    166.000    166.000    166.000
mean       1.051      0.982      1.091
std        0.286      0.143      0.317
min        0.622      0.654      0.367
25%        0.862      0.891      0.895
50%        0.994      0.950      1.040
75%        1.180      1.055      1.287
max        2.105      1.534      2.258
